# Replication notebook
This notebook walks through the package step by step: data loading, feature construction, sample splitting, model estimation, and evaluation.

In [1]:
from __future__ import annotations
from pathlib import Path

import logging
from pandas.api.types import is_numeric_dtype
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pandas import Timestamp
from typing import Any

from config import (
    CacheConfig,
    TimeframeConfig,
    DataFilesConfig,
    RunControlConfig,
    SplitConfig,
    DataRegimeConfig,
    HyperGridConfig,
    ReproducibilityConfig,
    CharacteristicsFrequency,
    LoggingConfig,
    ExpandingWindowConfig,
    ModelSelectionConfig,
)

from io_utils import ensure_dir, save_parquet, set_global_seed, setup_project_logger
from data_inputs import load_datashare, load_crsp_monthly, load_macro_monthly
from dataset_builder import impute_characteristics_by_month_cross_sectional_median, compute_missingness_for_characteristics, save_missingness_comparison_plot, summarize_columns, save_missingness_three_comparison_plot


In [2]:
repro_cfg = ReproducibilityConfig(random_state=42)
cache_cfg = CacheConfig()
tf_cfg = TimeframeConfig()
data_cfg = DataFilesConfig()
run_ctrl_cfg = RunControlConfig()
split_cfg = SplitConfig()
regime_cfg = DataRegimeConfig(mode="full")
grid_cfg = HyperGridConfig(use_extended_grids=False)
freq_cfg = CharacteristicsFrequency()
log_cfg = LoggingConfig()
expand_cfg = ExpandingWindowConfig()
model_cfg = ModelSelectionConfig()


In [3]:

# Set global seed ONCE at the start - ensures reproducibility across all runs
set_global_seed(repro_cfg.random_state, repro_cfg.torch_deterministic)

logger, log_path = setup_project_logger(
    logger_name=log_cfg.logger_name,
    log_dir=log_cfg.log_dir,
    regime_mode=regime_cfg.mode,
    overwrite_log=log_cfg.overwrite_log,
    file_level_full=log_cfg.file_level_full,
    file_level_coding=log_cfg.file_level_coding,
    console_level_full=log_cfg.console_level_full,
    console_level_coding=log_cfg.console_level_coding,
)

logger.info("Starting experiment run.")
logger.info(f"Random seed: {repro_cfg.random_state}")
logger.info(f"Cache dir: {cache_cfg.cache_dir}")
logger.info(f"Data regime: {regime_cfg.mode}")


2026-09-17 19:45:30 | INFO | eap_ml | Logger initialized.
2026-09-17 19:45:30 | INFO | eap_ml | Regime mode: full
2026-09-17 19:45:30 | INFO | eap_ml | Log file: logs\eap_ml_full_20260917_194530.log
2026-09-17 19:45:30 | INFO | eap_ml | Starting experiment run.
2026-09-17 19:45:30 | INFO | eap_ml | Random seed: 42
2026-09-17 19:45:30 | INFO | eap_ml | Cache dir: cache
2026-09-17 19:45:30 | INFO | eap_ml | Data regime: full


In [4]:
ensure_dir("output")
ensure_dir(cache_cfg.cache_dir)

In [5]:
complete_dataset_path=f"{cache_cfg.cache_dir}/complete_dataset.parquet"
descriptives_path=cache_cfg.descriptives_dir

feature_panel_path = f"{cache_cfg.cache_dir}/feature_panel_{regime_cfg.mode}.parquet"
feature_cols_path = f"{cache_cfg.cache_dir}/feature_cols_{regime_cfg.mode}.pkl"

In [6]:
datashare_path=data_cfg.datashare_path
crsp_path=data_cfg.crsp_monthly_path
macro_path=data_cfg.macro_path
out_path=complete_dataset_path
descriptives_path=descriptives_path
cache_enabled=cache_cfg.enabled
possible_crsp_cols=data_cfg.possible_crsp_cols
possible_marco_cols=data_cfg.possible_marco_cols
cols_chara=data_cfg.chara_cols
cols_vars_monthly=freq_cfg.cols_vars_monthly
cols_vars_quarterly=freq_cfg.cols_vars_quarterly
cols_vars_annual=freq_cfg.cols_vars_annual

In [7]:
logger = logging.getLogger("eap_ml.dataset_builder")

In [8]:
logger.info(f"Building complete dataset, output: {out_path}")


logger.debug("Loading source datasets")

crsp = load_crsp_monthly(crsp_path, possible_crsp_cols)
macro = load_macro_monthly(macro_path, possible_marco_cols)
ds = load_datashare(datashare_path)


2026-09-17 19:45:30 | INFO | eap_ml.dataset_builder | Building complete dataset, output: cache/complete_dataset.parquet
2026-09-17 19:45:30 | DEBUG | eap_ml.dataset_builder | Loading source datasets
2026-09-17 19:45:30 | DEBUG | eap_ml.data_inputs | Loading CRSP monthly data from C:/Coding/Project/data/crsp_monthly.csv
c:\Coding\thesis-project\data_inputs.py:120: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
2026-09-17 19:45:38 | DEBUG | eap_ml.data_inputs | Standardizing date of CRSP
2026-09-17 19:45:48 | DEBUG | eap_ml.data_inputs | CRSP loaded: 4594389 rows
2026-09-17 19:45:48 | DEBUG | eap_ml.data_inputs | Loading macro data from C:/Coding/Project/data/Data2024_monthly_goyal.csv
2026-09-17 19:45:48 | DEBUG | eap_ml.data_inputs | Standardizing date of Macro
2026-09-17 19:45:48 | DEBUG | eap_ml.data_inputs | Macro loaded: 1848 rows, columns: ['date', 'tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar'

In [9]:

#-------------
date_1987_05 = pd.Period("1987-05", freq="M")
#-------------
#-------------
test_1 = ds.loc[ds["date"] == date_1987_05].copy()
#-------------


In [10]:

# Columns 3-96 in datashare.csv = 94 characteristics.
characteristic_cols = cols_chara
logger.debug(f"Identified {len(characteristic_cols)} characteristic columns")

logger.debug("Merging datashare with CRSP")
merged = ds.merge(
    crsp[["permno", "date", "ret", "dlret"]],
    on=["permno", "date"],
    how="inner",
    # validate="one_to_one",
)


2026-09-17 19:47:14 | DEBUG | eap_ml.dataset_builder | Identified 94 characteristic columns
2026-09-17 19:47:14 | DEBUG | eap_ml.dataset_builder | Merging datashare with CRSP


In [11]:

#-------------
test_2 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

logger.debug("Merging with macro data")
merged = merged.merge(macro, on="date", how="left")

merged = merged.sort_values(["permno", "date"]).reset_index(drop=True)



2026-09-17 19:47:31 | DEBUG | eap_ml.dataset_builder | Merging with macro data


In [12]:

#-------------
test_3 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------


In [13]:

# ------------------------------------------------------------------
# ret_total
# ------------------------------------------------------------------
logger.debug("Creating ret_total")

has_return_data = (merged["ret"].notna() | merged["dlret"].notna())

merged["ret_total"] = (
    (1.0 + merged["ret"].fillna(0.0))
    * (1.0 + merged["dlret"].fillna(0.0))
    - 1.0
).where(has_return_data)    


2026-09-17 19:48:34 | DEBUG | eap_ml.dataset_builder | Creating ret_total


In [ ]:
#-------------
tracked_cols = ["ret_total", *cols_vars_monthly, *cols_vars_quarterly, *cols_vars_annual]
#-------------
test_4 = merged.loc[merged["date"] == date_1987_05].copy()
missing_4 = merged[tracked_cols].isna().copy()
#-------------


In [ ]:

# ------------------------------------------------------------------
# Missingness visualisation and imputation
# ------------------------------------------------------------------
cols_chara_and_ret_total = characteristic_cols + ["ret_total"]

# Visualisation before imputation
missing_before = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

before_csv = f"{descriptives_path}/charas_missingness_before.csv"
missing_before.to_csv(before_csv, index=False)

#-------------
test_5 = merged.loc[merged["date"] == date_1987_05].copy()
missing_5 = merged[tracked_cols].isna().copy()
#-------------


In [ ]:


logger.debug(f"Saved missingness before imputation: {before_csv}")

# Imputation
logger.info("Imputing missing characteristics using monthly cross-sectional medians")
merged = impute_characteristics_by_month_cross_sectional_median(merged, cols_chara_and_ret_total)

#-------------
test_6 = merged.loc[merged["date"] == date_1987_05].copy()
missing_6 = merged[tracked_cols].isna().copy()
#-------------

# Visualisation after imputation
missing_after = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

after_csv = f"{descriptives_path}/charas_missingness_after.csv"
missing_after.to_csv(after_csv, index=False)


logger.debug(f"Saved missingness after imputation: {after_csv}")

# Comparison plot
comparison_jpg = f"{descriptives_path}/charas_missingness_comparison.jpg"
save_missingness_comparison_plot(
    missing_before,
    missing_after,
    comparison_jpg,
    title="Missing Data Percentage: Before vs After Imputation",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")



2026-09-17 19:48:40 | DEBUG | eap_ml.dataset_builder | Saved missingness before imputation: descriptives/charas_missingness_before.csv
2026-09-17 19:48:40 | INFO | eap_ml.dataset_builder | Imputing missing characteristics using monthly cross-sectional medians
2026-09-17 19:49:39 | DEBUG | eap_ml.dataset_builder | Saved missingness after imputation: descriptives/charas_missingness_after.csv
2026-09-17 19:49:41 | DEBUG | eap_ml.dataset_builder | Saved missingness comparison plot: descriptives/charas_missingness_comparison.jpg


In [ ]:

# ------------------------------------------------------------------
# Temporal shifts 
# ------------------------------------------------------------------


#-------------
test_7 = merged.loc[merged["date"] == date_1987_05].copy()
missing_7 = merged[tracked_cols].isna().copy()
#-------------


In [ ]:

# Build next-month excess return target.
logger.debug("Building lead return target (shift ret_total)")


# Build shifts for monthly, quarterly and annual characteristcs
logger.debug("Building characteristic shifts and ret_total shift")
merged = merged.sort_values(["permno", "date"]).copy()
grouped = merged.groupby("permno")

for i in merged.columns:
    if i in cols_vars_monthly or i == "ret_total":
        merged[i] = merged.groupby("permno")[i].shift(-1)
    elif i in cols_vars_quarterly:
        merged[i] = merged.groupby("permno")[i].shift(-3)
    elif i in cols_vars_annual:
        merged[i] = merged.groupby("permno")[i].shift(-6)


#-------------
test_8 = merged.loc[merged["date"] == date_1987_05].copy()
missing_8 = merged[tracked_cols].isna().copy()
#-------------


2026-09-17 19:49:42 | DEBUG | eap_ml.dataset_builder | Building lead return target (shift ret_total)
2026-09-17 19:49:42 | DEBUG | eap_ml.dataset_builder | Building characteristic shifts and ret_total shift


In [19]:

# ------------------------------------------------------------------
# Summary 
# ------------------------------------------------------------------
summary_before = summarize_columns(merged, cols_chara_and_ret_total)

summary_before_csv = f"{descriptives_path}/summary_before.csv"
summary_before.to_csv(summary_before_csv, index=False)

# Define inclusive monthly boundaries
start_period = pd.Period("1957-10", freq="M")
end_period = pd.Period("2021-06", freq="M")

# Keep observations from October 1957 through June 2021

mask = (
    (merged["date"] >= start_period)
    & (merged["date"] <= end_period)
)
merged = merged.loc[mask].copy()

logger.debug(f"Dataset reduced due to missing values to : {merged["date"].min()} and {merged["date"].max()}")



2026-09-17 19:51:41 | DEBUG | eap_ml.dataset_builder | Dataset reduced due to missing values to : 1957-10 and 2021-06


In [ ]:

#-------------
test_9 = merged.loc[merged["date"] == date_1987_05].copy()
missing_9 = merged[tracked_cols].isna().copy()
#-------------


In [21]:

# Visualisation of missingness after temporal cuts
missing_after_cut = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

after_cut_csv = f"{descriptives_path}/charas_missingness_after_cut.csv"
missing_after_cut.to_csv(after_cut_csv, index=False)

logger.debug(f"Saved missingness after temporal cut: {after_cut_csv}")

# Comparison plot
comparison_three_jpg = f"{descriptives_path}/charas_missingness_comparison_three.jpg"
save_missingness_three_comparison_plot(
    missing_before,
    missing_after,
    missing_after_cut,
    comparison_three_jpg,
    title="Missing Data Percentage: Before vs After Imputation vs After Temporal Reduction",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")



2026-09-17 19:51:48 | DEBUG | eap_ml.dataset_builder | Saved missingness after temporal cut: descriptives/charas_missingness_after_cut.csv
2026-09-17 19:51:49 | DEBUG | eap_ml.dataset_builder | Saved missingness comparison plot: descriptives/charas_missingness_comparison.jpg


In [ ]:

#-------------
test_10 = merged.loc[merged["date"] == date_1987_05].copy()
missing_10 = merged[tracked_cols].isna().copy()
#-------------


In [ ]:
#-------------
test_1_csv = f"{descriptives_path}/test_1.csv"
test_1.to_csv(test_1_csv, index=False)

test_2_csv = f"{descriptives_path}/test_2.csv"
test_2.to_csv(test_2_csv, index=False)

test_3_csv = f"{descriptives_path}/test_3.csv"
test_3.to_csv(test_3_csv, index=False)

test_4_csv = f"{descriptives_path}/test_4.csv"
test_4.to_csv(test_4_csv, index=False)

test_5_csv = f"{descriptives_path}/test_5.csv"
test_5.to_csv(test_5_csv, index=False)

test_6_csv = f"{descriptives_path}/test_6.csv"
test_6.to_csv(test_6_csv, index=False)

test_7_csv = f"{descriptives_path}/test_7.csv"
test_7.to_csv(test_7_csv, index=False)

test_8_csv = f"{descriptives_path}/test_8.csv"
test_8.to_csv(test_8_csv, index=False)

test_9_csv = f"{descriptives_path}/test_9.csv"
test_9.to_csv(test_9_csv, index=False)

test_10_csv = f"{descriptives_path}/test_9.csv"
test_10.to_csv(test_9_csv, index=False)
#-------------
newly_missing_5 = (
    ~missing_4
    & missing_5
)



newly_missing_rows_5 = merged.loc[
    newly_missing5.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print(newly_missing_rows_5.head(20))


In [ ]:


logger.info(f"Complete dataset built: {merged.shape}")
save_parquet(merged, out_path, enabled=cache_enabled)

In [ ]:
merged["date"] = pd.to_datetime(merged["date"])



for i in merged.columns:
    if i in cols_vars_monthly:
        new_column = (merged.set_index("date")
            .groupby("permno")[i]
            .shift(-1, freq=pd.DateOffset(months=1)))
        merged.set_index(["permno", "date"], inplace=True)
        merged[i] = new_column
        merged.reset_index(inplace=True)
    elif i in cols_vars_quarterly:
        new_column = (merged.set_index("date")
            .groupby("permno")[i]
            .shift(-1, freq=pd.DateOffset(months=1)))
        merged.set_index(["permno", "date"], inplace=True)
        merged[i] = new_column
        merged.reset_index(inplace=True)
    elif i in cols_vars_annual:
        new_column = (merged.set_index("date")
            .groupby("permno")[i]
            .shift(-1, freq=pd.DateOffset(months=1)))
        merged.set_index(["permno", "date"], inplace=True)
        merged[i] = new_column
        merged.reset_index(inplace=True)


"""
new_column = (merged.set_index("date")
                .groupby("permno")[i]
                .shift(-1, freq=pd.DateOffset(months=1)))
merged.set_index(["permno", "date"], inplace=True)
merged[i] = new_column
merged.reset_index(inplace=True)
"""

merged["date"] = merged["date"].dt.to_period("M")

In [ ]:
tracked_cols = ["ret_total", *cols_vars_monthly, *cols_vars_quarterly, *cols_vars_annual]
missing_before = merged[tracked_cols].isna().copy()



missing_after = merged[tracked_cols].isna()
newly_missing = (
    ~missing_before
    & missing_after
)



newly_missing_rows = merged.loc[
    newly_missing.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print(newly_missing_rows.head(20))

In [ ]:
merged["date"] = pd.to_datetime(merged["date"])
new_column = (merged.set_index("date")
                .groupby("permno")[i]
                .shift(-1, freq=pd.DateOffset(months=1)))
merged.set_index(["permno", "date"], inplace=True)
merged[i] = new_column
merged.reset_index(inplace=True)
merged["date"] = merged["date"].dt.to_period("M")